# StealTheDealAI: Model Comparison (local)

Runs entirely on this machine. Scores the DNN, Specialist (fine-tuned LLM on Modal), and Frontier (RAG) agents - plus the weighted ensemble - against a held-out test split, so you can see which is actually pulling its weight before tuning `ENSEMBLE_WEIGHT_*` in `config/settings.py`.

**Prerequisites** (whatever you've completed so far - missing pieces are skipped gracefully, not fatal):
- `data/processed/training_data.csv` exists (ran `scripts/clean_training_data.py`)
- `models/deep_neural_network.pth` + `models/dnn_norm_stats.json` exist (ran notebook 02 on Kaggle)
- `vectorstore/products_vectorstore/` exists (ran `scripts/build_vector_store.py`)
- `modal deploy modal_deployments/pricer_service.py` has been run (for the Specialist agent)
- `NIM_API_KEY` set in `config/.env` (for the Frontier agent, which calls an LLM)

In [1]:
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd().parent))
from config import settings
from agents.items import Item
from agents.frontier_agent import FrontierAgent
from agents.specialist_agent import SpecialistAgent
from agents.neural_network_agent import NeuralNetworkAgent
from agents import evaluator

SAMPLE_SIZE = 60  # test rows to score - each hits the NIM API and (if deployed) Modal, so keep this modest

c:\Users\yadav\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Load the held-out test split

Re-derives the same train/val/test split `scripts/prepare_finetune_jsonl.py` used (same `random_state=42`, same `settings.FINETUNE_SPLIT` fractions) directly on `training_data.csv`, so we get the held-out rows back with their structured fields intact (rather than re-parsing `finetune_test.jsonl`'s prompt text).

In [2]:
TRAINING_DATA_PATH = settings.DATA_PROCESSED_DIR / "training_data.csv"

if not TRAINING_DATA_PATH.exists():
    raise FileNotFoundError(f"{TRAINING_DATA_PATH} not found - run scripts/clean_training_data.py first.")

df = pd.read_csv(TRAINING_DATA_PATH).dropna(subset=["title", "description", "price"])

train_frac, val_frac, test_frac = settings.FINETUNE_SPLIT
train_df, temp_df = train_test_split(df, test_size=(val_frac + test_frac), random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=test_frac / (val_frac + test_frac), random_state=42)

sample_df = test_df.sample(min(SAMPLE_SIZE, len(test_df)), random_state=42)
print(f"Test split: {len(test_df)} rows total, scoring a sample of {len(sample_df)}")

Test split: 2934 rows total, scoring a sample of 60


## Initialize agents

Each agent already degrades gracefully (`estimate()` returns `None`) if its dependency isn't ready yet - e.g. Specialist if Modal isn't deployed, DNN if `models/` is empty.

In [3]:
frontier = FrontierAgent()
specialist = SpecialistAgent()
neural_net = NeuralNetworkAgent()

AGENTS = {"DNN": neural_net, "Specialist": specialist, "Frontier (RAG)": frontier}
WEIGHTS = {"DNN": settings.ENSEMBLE_WEIGHT_DNN, "Specialist": settings.ENSEMBLE_WEIGHT_SPECIALIST, "Frontier (RAG)": settings.ENSEMBLE_WEIGHT_RAG}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Score each test row

Builds an `Item` directly from the already-clean CSV fields (skips the LLM preprocessing step - we're scoring the price estimators, not the extraction step) and queries all three agents concurrently per row, same as `EnsembleAgent.process()` does.

In [4]:
def score_row(row):
    item = Item(
        product_title=str(row["title"])[:200],
        product_category=str(row["category"])[:100],
        product_description=str(row["description"])[:1000],
        product_price=float(row["price"]),
    )
    result = {"label": item.product_title, "actual": item.product_price}
    with ThreadPoolExecutor(max_workers=len(AGENTS)) as ex:
        futures = {ex.submit(agent.estimate, item): name for name, agent in AGENTS.items()}
        for future, name in futures.items():
            try:
                result[name] = future.result()
            except Exception as e:
                print(f"{name} raised on '{item.product_title[:40]}': {e}")
                result[name] = None

    weighted_sum, weight_total = 0.0, 0.0
    for name, weight in WEIGHTS.items():
        if result[name] is not None:
            weighted_sum += result[name] * weight
            weight_total += weight
    result["Ensemble"] = weighted_sum / weight_total if weight_total > 0 else None
    return result

results = [score_row(row) for _, row in sample_df.iterrows()]
results_df = pd.DataFrame(results)
results_df.head(10)

,label,actual,DNN,Specialist,Frontier (RAG),Ensemble
0,"Midiron Gift for Sister, Coffee Mug, Teddy wit...",1699.0,1698.395956,1599.0,899.0,1398.728180
1,Royal Beauty China : SOON PURE Black Gold Aqua...,4786.0,4826.869955,3999.0,449.0,3129.041480
2,HANSKIN Ceramide Moisture Eye Cream 24ml 0.84o...,4356.0,4188.306730,1999.0,899.0,2599.188029
3,Pink Root Hair Serum and Streax Regular Light ...,439.0,643.279635,3999.0,549.0,1281.425836
4,World Beautys New Korean Cosmetics PUREBESS Sn...,1690.0,1610.561905,1999.0,449.0,1281.702857
5,"Urban Platter Black Pepper Banana Chips, 400g",300.0,1720.636584,250.0,299.0,928.936463
6,"axe shower gel, snake peel, 16 fluid ounce (pa...",6659.0,3082.959726,3999.0,599.0,2396.781877
7,KKY 12Pcs Multi Color Design Gel Pens Massage ...,2679.0,1332.007847,999.0,499.0,973.853531
8,World Beautys New Goji Cream Whitening face cr...,1255.0,1691.519236,10200.0,349.0,2923.333656
9,SLB Works 10pcs Cosmetic Spatula Plastic Home ...,1475.0,1796.604413,1390.0,249.0,1173.621986


## Per-model charts and metrics

In [5]:
metrics_table = {}

for model_name in ["DNN", "Specialist", "Frontier (RAG)", "Ensemble"]:
    scored = results_df.dropna(subset=[model_name])
    if len(scored) < 2:
        print(f"{model_name}: not enough predictions to score ({len(scored)}/{len(results_df)}) - skipping.")
        continue

    scatter_fig, trend_fig, metrics = evaluator.report(
        actual=scored["actual"].tolist(),
        predicted=scored[model_name].tolist(),
        labels=scored["label"].tolist(),
        title=model_name,
    )
    metrics["n"] = len(scored)
    metrics_table[model_name] = metrics
    trend_fig.show()
    scatter_fig.show()

## Summary comparison

Use this to re-tune `ENSEMBLE_WEIGHT_RAG` / `ENSEMBLE_WEIGHT_SPECIALIST` / `ENSEMBLE_WEIGHT_DNN` in `config/settings.py` - the model(s) with the lowest MAE/RMSE and highest r² deserve more weight.

In [6]:
summary = pd.DataFrame(metrics_table).T[["n", "mae", "rmse", "r2"]]
summary.columns = ["n scored", "MAE (₹)", "RMSE (₹)", "r² (%)"]
summary.round(1)

,n scored,MAE (₹),RMSE (₹),r² (%)
DNN,60.0,947.9,1930.4,55.7
Specialist,60.0,1434.0,2463.2,27.9
Frontier (RAG),60.0,2419.6,3612.5,-55.1
Ensemble,60.0,1317.1,2184.3,43.3
